# Lab: Efficient Storage and Serialisation of a Priority Queue

A priority queue (here: a `heapq` heap) has to be **saved and restored**, e.g. to resume a long search or to share a task backlog.

This lab uses **NYC 311 service requests** as a realistic backlog: each request is an item, the oldest request has the highest priority.

Main goals:

- store a heap of tuples in `Pickle`, `JSON`, `Parquet` and NumPy `NPZ`
- measure file size, write time, and read time
- check that the heap invariant survives the round trip
- decide which format fits which task

## 1) Setup

For this lab, you may need:

```bash
pip install pandas numpy pyarrow matplotlib
```

In [ ]:
import heapq
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### 1.1) Download the dataset

We fetch 100,000 recent requests from the [NYC Open Data 311 API](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9) and cache them locally as CSV.

In [ ]:
DATA_DIR = Path("storage_lab_data")
DATA_DIR.mkdir(exist_ok=True)

url = (
    "https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
    "?$limit=100000&$select=unique_key,created_date,agency,complaint_type,borough"
)
source = DATA_DIR / "nyc_311_sample.csv"

if not source.exists():
    pd.read_csv(url).to_csv(source, index=False)

raw = pd.read_csv(source, parse_dates=["created_date"]).dropna()
print(raw.shape)
raw.head()

Full shape: (2463931, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.0,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.5,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0


### 1.2) Build the priority queue

Each heap entry is a tuple `(priority, unique_key, agency, complaint_type, borough)`.
The priority is the creation time in seconds (older request = smaller value = served first).
`unique_key` breaks ties, just like a counter in your own priority queue.

In [ ]:
priority = raw["created_date"].astype("int64") // 10**9

heap = list(
    zip(
        priority.tolist(),
        raw["unique_key"].tolist(),
        raw["agency"].tolist(),
        raw["complaint_type"].tolist(),
        raw["borough"].tolist(),
    )
)
heapq.heapify(heap)

print(len(heap), heap[0])
print(
    "heap invariant holds:",
    all(
        heap[i] <= heap[c]
        for i in range(len(heap))
        for c in (2 * i + 1, 2 * i + 2)
        if c < len(heap)
    ),
)

(100000, 15)


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
0,2022-01-15 00:22:49,2022-01-15 00:31:33,2.0,1.50,1,N,148,209,1,8.0,3.0,0.5,2.36,0.0,14.16
1,2022-01-13 17:50:32,2022-01-13 18:03:09,1.0,2.60,1,N,48,238,1,11.0,3.5,0.5,1.00,0.0,16.30
2,2022-01-19 18:41:39,2022-01-19 18:56:29,1.0,3.73,1,N,141,24,1,13.5,1.0,0.5,3.56,0.0,21.36
3,2022-01-08 10:35:44,2022-01-08 10:45:53,1.0,5.96,1,N,231,140,1,17.5,0.0,0.5,4.16,0.0,24.96
4,2022-01-26 20:08:13,2022-01-26 20:12:15,2.0,0.69,1,N,229,162,2,5.0,0.5,0.5,0.00,0.0,8.80


## 2) Exercises

For each format: save the `heap`, load it back, and record write time, read time and file size.

### 2.0) Helper functions

Implement helpers that return the execution time of a call and the size of a file in MB. Use them in every exercise below and collect the results in one dict `results`.

### 2.1) Pickle

"Pickle" the `heap` (the list itself, so the array order is kept).

### 2.2) JSON

Save the `heap` as `.json`. What happens to the tuples when you load it back?

### 2.3) Parquet

Convert the `heap` to a `DataFrame` (one column per tuple field, keep the heap order) and save it as `.parquet`.

### 2.4) NPZ

Strings cannot be stored efficiently in a plain numeric array. Save only `priority` and `unique_key` (in heap order) as `.npz`. What do you lose?

### 2.5) Compare results

Plot file size, write time and read time of all formats (e.g. three bar charts).

## 3) Observation and Discussion

### 3.1) Is the loaded data still a valid heap?

For each format, load the data back, rebuild a list of tuples if needed, and check that it equals the original `heap` and that the heap invariant holds. Which formats need a `heapq.heapify` after loading? What does that cost?

### 3.2) Partial loading

`Parquet` supports column-wise reading. Read only the `priority` and `unique_key` columns and compare the read time with loading everything.

### 3.3) Take-home lessons

Fill in this section based on your observations and understanding of the topic.